In [12]:
from __future__ import annotations
import math, shutil, subprocess, warnings
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.collections import LineCollection
from PIL import Image

warnings.filterwarnings('ignore')
torch.set_default_dtype(torch.float64)

# =============================================================================
# PARAMETERS
# =============================================================================
TARGET = np.array([4.5, 3.0], float)
COOKIE_START = np.array([0.0, 0.0], float)
ANT_INIT_R = 2.0
N_ANTS = 45

EPS = 0.12
ELL = 1.2
KAPPA = 4.0
C_SAT = 5.0
BETA_SEEK = 3.0
BETA_HOLD = 2.5
BETA_PULL = 3.0
U_MAX = 2.0
W0 = 0.35
SIGMA_REP = 0.8
F0 = 5.0
GAMMA_C = 3.0
SIGMA_X = 0.22
SIGMA_Z = 0.04

# Time compression so the same transport phenomenon happens on [0, 10].
# Roughly, the original learned policy reached the target around t≈21-22.
# Multiplying drift by c and diffusion by sqrt(c) corresponds to a time change.
TIME_SCALE = 2.2

ROCK_POS = np.array([2.2, 1.5], float)
ROCK_RADIUS = 0.65
ROCK_REP_SIGMA = 0.80
ROCK_REP_AMP_ANT = 8.0
ROCK_REP_AMP_COOKIE = 20.0

# Reward / objective
C_Y = 5.0
C_U = 0.05
C_ETA = 0.05
C_D = 0.15
LAMBDA_ENT = 0.05
BETA_DISC = 0.30

# Policy omega=[seek,pull,eta]. Fixed latent Gaussian variances.
OMEGA_INIT = np.array([0.2, 0.2, 0.2], float)
SIGMA_U_RAW = 0.45
SIGMA_ETA_RAW = 0.45
ETA_DENSITY_EPS = 1e-5

# MF-PhiBE training data on [0, T_DATA]
N_POLICY_ITER = 18
L_EPS_DATA = 4
T_DATA = 10.0
DT_DATA = 0.05
N_STEPS_DATA = int(round(T_DATA / DT_DATA))
N_PART_PHIBE = 12
ENV_SUBSTEPS_PER_OBS = 5
USE_DISCOUNTED_WEIGHTS = True
CRITIC_RIDGE = 1e-6

# PPO
PPO_CLIP_EPS = 0.20
PPO_LR = 0.02
PPO_EPOCHS = 10
PPO_MINIBATCH = 2048
PPO_MAX_GRAD_NORM = 0.50
PPO_TARGET_KL = 0.03
CENTER_ADV_BY_LAW = True
NORMALIZE_ADV_GLOBALLY = False

# Cylindrical moments
POS_SCALE = 5.0
ROCK_FEATURE_H_COOKIE = 1.25
ROCK_FEATURE_H_ANT = 1.25
PATH_VEC = TARGET - COOKIE_START
PATH_DIR = PATH_VEC / np.linalg.norm(PATH_VEC)
PATH_PERP = np.array([-PATH_DIR[1], PATH_DIR[0]])
N_MOMENTS = 9
N_CRITIC_BASIS = 1 + N_MOMENTS + N_MOMENTS * (N_MOMENTS + 1) // 2

# Animation
T_ANIM = 10.0
DT_ANIM = 0.025
FPS_FULL = 10
FPS_SIMPLE = 12
SIM_PER_FRAME = 4
SEED_ANIM = 2024
SEED_DATA = 42
RUN_DERIVATIVE_CHECKS = True

OUT_DIR = Path('images_obstacle_mfphibe_ppo_T10')
FULL_FRAME_DIR = Path('frame_images_obstacle_mfphibe_ppo_T10_full')
SIMPLE_FRAME_DIR = Path('frame_images_obstacle_mfphibe_ppo_T10_simple')
for d in [OUT_DIR, FULL_FRAME_DIR, SIMPLE_FRAME_DIR]:
    d.mkdir(exist_ok=True)

COL_BG = '#FFFFFF'
COL_GRID = '#FFFFFF'
COL_COOKIE = '#C8860A'
COL_CRUMB = '#8B5E0A'
COL_TRAIL = '#A07030'
COL_TARGET = '#1F4E79'
CMAP_ANT = plt.cm.RdYlBu_r
X_MIN, X_MAX = -2.5, 6.5
Y_MIN, Y_MAX = -2.0, 5.2

# =============================================================================
# BASIC UTILITIES
# =============================================================================
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -40.0, 40.0)))

def Lambda(z):
    return sigmoid(z)

def psi_fn(d):
    return np.exp(-(d / ELL) ** 2)

def logit(p):
    p = np.clip(p, ETA_DENSITY_EPS, 1.0 - ETA_DENSITY_EPS)
    return np.log(p) - np.log1p(-p)

def normal_logpdf_np(x, mean, std):
    return -0.5 * ((x - mean) / std) ** 2 - math.log(std) - 0.5 * math.log(2 * math.pi)

def clean_frame_dir(path: Path):
    for p in path.glob('frame_*.png'):
        p.unlink()
    lst = path / 'list.txt'
    if lst.exists():
        lst.unlink()

# =============================================================================
# ENVIRONMENT
# =============================================================================
def rock_force(pos, amp):
    scalar = pos.ndim == 1
    p = pos[None, :] if scalar else pos
    d = p - ROCK_POS[None, :]
    w = np.exp(-np.sum(d * d, axis=1) / (2 * ROCK_REP_SIGMA ** 2))
    f = (amp / ROCK_REP_SIGMA ** 2) * d * w[:, None]
    return f[0] if scalar else f

def repulsion_all(x):
    M = x.shape[0]
    d = x[:, None, :] - x[None, :, :]
    w = np.exp(-np.sum(d * d, axis=2) / (2 * SIGMA_REP ** 2))
    np.fill_diagonal(w, 0.0)
    return (W0 / SIGMA_REP ** 2) * np.sum(d * w[:, :, None], axis=1) / M

def mf_force(x, z, y):
    d = x - y[None, :]
    r = np.linalg.norm(d, axis=1)
    unit = d / (r[:, None] + EPS)
    return F0 * np.mean(Lambda(z)[:, None] * psi_fn(r)[:, None] * unit, axis=0)

def drift_x_all(x, z, y, u, rep):
    lam = Lambda(z)
    dc = y[None, :] - x
    r = np.linalg.norm(dc, axis=1)
    dseek = dc / (r[:, None] + EPS)
    tv = TARGET - y
    dpull = tv / (np.linalg.norm(tv) + EPS)
    return (
        u + rep
        + BETA_SEEK * (1 - lam)[:, None] * psi_fn(r)[:, None] * dseek
        - BETA_HOLD * lam[:, None] * (x - y[None, :])
        + BETA_PULL * lam[:, None] * dpull[None, :]
        + rock_force(x, ROCK_REP_AMP_ANT)
    )

def drift_z_all(x, z, y, eta):
    r = np.linalg.norm(x - y[None, :], axis=1)
    return KAPPA * (C_SAT * eta * psi_fn(r) - z)

def running_reward_all(x, z, y, u, eta):
    rc = -0.5 * C_Y * np.sum((y - TARGET) ** 2)
    return (
        rc
        - 0.5 * C_U * np.sum(u * u, axis=1)
        - 0.5 * C_ETA * eta ** 2
        - 0.5 * C_D * np.sum((x - y[None, :]) ** 2, axis=1)
    )

def environment_substep(x, z, y, u, eta, rng, dt):
    bx = drift_x_all(x, z, y, u, repulsion_all(x))
    bz = drift_z_all(x, z, y, eta)
    by = (mf_force(x, z, y) + rock_force(y, ROCK_REP_AMP_COOKIE)) / GAMMA_C
    c = TIME_SCALE
    xn = x + c * bx * dt + rng.normal(0, math.sqrt(c * dt) * SIGMA_X, size=x.shape)
    zn = z + c * bz * dt + rng.normal(0, math.sqrt(c * dt) * SIGMA_Z, size=z.shape)
    yn = y + c * by * dt
    return xn, zn, yn

def advance_occupancy(x, z, y, u, eta, rng, Delta):
    dt = Delta / ENV_SUBSTEPS_PER_OBS
    for _ in range(ENV_SUBSTEPS_PER_OBS):
        x, z, y = environment_substep(x, z, y, u, eta, rng, dt)
    return x, z, y

# -- strict local frozen-action branches ---------------------------------------
def _batched_repulsion(xp):
    B, M, _ = xp.shape
    d = xp[:, :, None, :] - xp[:, None, :, :]
    w = np.exp(-np.sum(d * d, axis=3) / (2 * SIGMA_REP ** 2))
    ii = np.arange(M)
    w[:, ii, ii] = 0.0
    return (W0 / SIGMA_REP ** 2) * np.sum(d * w[..., None], axis=2) / M

def _query_repulsion(xq, xp):
    d = xq[:, None, :] - xp
    w = np.exp(-np.sum(d * d, axis=2) / (2 * SIGMA_REP ** 2))
    return (W0 / SIGMA_REP ** 2) * np.mean(d * w[..., None], axis=1)

def _batched_force(xp, zp, yp):
    d = xp - yp[:, None, :]
    r = np.linalg.norm(d, axis=2)
    return F0 * np.mean(Lambda(zp)[..., None] * psi_fn(r)[..., None] * d / (r[..., None] + EPS), axis=1)

def local_frozen_action_branches(x, z, y, u, eta, rng, Delta):
    """For tuple j, freeze a_j over [t,t+Delta] in an independent branch."""
    M = x.shape[0]
    B = M
    dt = Delta / ENV_SUBSTEPS_PER_OBS
    c = TIME_SCALE

    xp = np.repeat(x[None, :, :], B, axis=0)
    zp = np.repeat(z[None, :], B, axis=0)
    yp = np.repeat(y[None, :], B, axis=0)
    xr = x.copy()
    zr = z.copy()

    for _ in range(ENV_SUBSTEPS_PER_OBS):
        rep = _batched_repulsion(xp)
        lam = Lambda(zp)
        dc = yp[:, None, :] - xp
        r = np.linalg.norm(dc, axis=2)
        ds = dc / (r[..., None] + EPS)
        tv = TARGET[None, :] - yp
        dp = tv / (np.linalg.norm(tv, axis=1)[:, None] + EPS)
        bx = (
            u[:, None, :] + rep
            + BETA_SEEK * (1 - lam)[..., None] * psi_fn(r)[..., None] * ds
            - BETA_HOLD * lam[..., None] * (xp - yp[:, None, :])
            + BETA_PULL * lam[..., None] * dp[:, None, :]
            + rock_force(xp.reshape(-1, 2), ROCK_REP_AMP_ANT).reshape(xp.shape)
        )
        bz = KAPPA * (C_SAT * eta[:, None] * psi_fn(r) - zp)
        by = (_batched_force(xp, zp, yp) + rock_force(yp, ROCK_REP_AMP_COOKIE)) / GAMMA_C

        rq = _query_repulsion(xr, xp)
        l = Lambda(zr)
        dcq = yp - xr
        rr = np.linalg.norm(dcq, axis=1)
        dsq = dcq / (rr[:, None] + EPS)
        bxr = (
            u + rq
            + BETA_SEEK * (1 - l)[:, None] * psi_fn(rr)[:, None] * dsq
            - BETA_HOLD * l[:, None] * (xr - yp)
            + BETA_PULL * l[:, None] * dp
            + rock_force(xr, ROCK_REP_AMP_ANT)
        )
        bzr = KAPPA * (C_SAT * eta * psi_fn(rr) - zr)

        xp = xp + c * bx * dt + rng.normal(0, math.sqrt(c * dt) * SIGMA_X, size=xp.shape)
        zp = zp + c * bz * dt + rng.normal(0, math.sqrt(c * dt) * SIGMA_Z, size=zp.shape)
        xr = xr + c * bxr * dt + rng.normal(0, math.sqrt(c * dt) * SIGMA_X, size=xr.shape)
        zr = zr + c * bzr * dt + rng.normal(0, math.sqrt(c * dt) * SIGMA_Z, size=zr.shape)
        yp = yp + c * by * dt

    return np.column_stack([xr, zr, yp])

# =============================================================================
# POLICY
# =============================================================================
def policy_latent_means_np(x, y, omega):
    dc = y[None, :] - x
    r = np.linalg.norm(dc, axis=1)
    ds = dc / (r[:, None] + EPS)
    tv = TARGET - y
    dp = tv / (np.linalg.norm(tv) + EPS)
    mu_u = omega[0] * ds + omega[1] * dp[None, :]
    eta_med = sigmoid(omega[2]) * psi_fn(r)
    mu_e = logit(eta_med)
    return mu_u, mu_e

def radial_squash(v):
    return U_MAX * v / np.sqrt(1 + np.sum(v * v, axis=1, keepdims=True))

def radial_logdet(v):
    return 2 * math.log(U_MAX) - 2 * np.log1p(np.sum(v * v, axis=1))

def sample_policy_np(x, z, y, omega, rng):
    mu_u, mu_e = policy_latent_means_np(x, y, omega)
    ru = mu_u + rng.normal(0, SIGMA_U_RAW, size=mu_u.shape)
    re = mu_e + rng.normal(0, SIGMA_ETA_RAW, size=mu_e.shape)
    u = radial_squash(ru)
    eta = sigmoid(re)
    lb_u = np.sum(normal_logpdf_np(ru, mu_u, SIGMA_U_RAW), axis=1)
    lb_e = normal_logpdf_np(re, mu_e, SIGMA_ETA_RAW)
    lj_e = np.log(np.clip(eta * (1 - eta), 1e-14, None))
    logp = (lb_u - radial_logdet(ru)) + (lb_e - lj_e)
    return u, eta, ru, re, logp

def torch_latent_means(states, omega):
    x = states[:, :2]
    y = states[:, 3:5]
    dc = y - x
    r = torch.linalg.vector_norm(dc, dim=1)
    ds = dc / (r[:, None] + EPS)
    tgt = torch.as_tensor(TARGET, dtype=states.dtype, device=states.device)
    tv = tgt[None, :] - y
    dp = tv / (torch.linalg.vector_norm(tv, dim=1)[:, None] + EPS)
    mu_u = omega[0] * ds + omega[1] * dp
    med = torch.sigmoid(omega[2]) * torch.exp(-(r / ELL) ** 2)
    med = torch.clamp(med, ETA_DENSITY_EPS, 1 - ETA_DENSITY_EPS)
    mu_e = torch.log(med) - torch.log1p(-med)
    return mu_u, mu_e

def torch_logp(states, ru, re, omega):
    mu_u, mu_e = torch_latent_means(states, omega)
    lbu = (-0.5 * ((ru - mu_u) / SIGMA_U_RAW) ** 2 - math.log(SIGMA_U_RAW) - 0.5 * math.log(2 * math.pi)).sum(dim=1)
    lju = 2 * math.log(U_MAX) - 2 * torch.log1p((ru * ru).sum(dim=1))
    lbe = -0.5 * ((re - mu_e) / SIGMA_ETA_RAW) ** 2 - math.log(SIGMA_ETA_RAW) - 0.5 * math.log(2 * math.pi)
    et = torch.sigmoid(re)
    lje = torch.log(torch.clamp(et * (1 - et), min=1e-14))
    return (lbu - lju) + (lbe - lje)

def torch_kl(states, old, new):
    ou, oe = torch_latent_means(states, old)
    nu, ne = torch_latent_means(states, new)
    return 0.5 * ((ou - nu) ** 2).sum(dim=1) / SIGMA_U_RAW ** 2 + 0.5 * (oe - ne) ** 2 / SIGMA_ETA_RAW ** 2

# =============================================================================
# CYLINDRICAL FEATURES AND ANALYTIC LIONS DERIVATIVES
# =============================================================================
def zeta_values_grad_hess(states):
    s = np.atleast_2d(states)
    N = s.shape[0]
    x = s[:, :2]
    z = s[:, 2]
    y = s[:, 3:5]

    val = np.zeros((N, N_MOMENTS))
    grad = np.zeros((N, N_MOMENTS, 5))
    H = np.zeros((N, N_MOMENTS, 5, 5))
    S = POS_SCALE
    S2 = S * S

    # 0. cookie-target distance^2
    d = y - TARGET
    val[:, 0] = np.sum(d * d, axis=1) / S2
    grad[:, 0, 3:5] = 2 * d / S2
    H[:, 0, 3, 3] = 2 / S2
    H[:, 0, 4, 4] = 2 / S2

    # 1. ant-cookie distance^2
    d = x - y
    val[:, 1] = np.sum(d * d, axis=1) / S2
    grad[:, 1, :2] = 2 * d / S2
    grad[:, 1, 3:5] = -2 * d / S2
    for j in range(2):
        H[:, 1, j, j] = 2 / S2
        H[:, 1, 3 + j, 3 + j] = 2 / S2
        H[:, 1, j, 3 + j] = -2 / S2
        H[:, 1, 3 + j, j] = -2 / S2

    # 2. attachment level Lambda(z)
    l = Lambda(z)
    lp = l * (1 - l)
    lpp = lp * (1 - 2 * l)
    val[:, 2] = l
    grad[:, 2, 2] = lp
    H[:, 2, 2, 2] = lpp

    # 3-4. attached signed displacement
    for j, k in [(0, 3), (1, 4)]:
        dj = d[:, j]
        val[:, k] = l * dj / S
        grad[:, k, j] = l / S
        grad[:, k, 3 + j] = -l / S
        grad[:, k, 2] = lp * dj / S
        H[:, k, 2, 2] = lpp * dj / S
        H[:, k, j, 2] = lp / S
        H[:, k, 2, j] = lp / S
        H[:, k, 3 + j, 2] = -lp / S
        H[:, k, 2, 3 + j] = -lp / S

    # 5-6. cookie progress / cross-track relative to the nominal path
    yc = y - COOKIE_START
    val[:, 5] = (yc @ PATH_DIR) / S
    grad[:, 5, 3:5] = PATH_DIR / S
    val[:, 6] = (yc @ PATH_PERP) / S
    grad[:, 6, 3:5] = PATH_PERP / S

    # 7. cookie-rock Gaussian
    eye = np.eye(2)
    d = y - ROCK_POS
    h = ROCK_FEATURE_H_COOKIE
    g = np.exp(-np.sum(d * d, axis=1) / (2 * h * h))
    val[:, 7] = g
    grad[:, 7, 3:5] = -g[:, None] * d / h ** 2
    H[:, 7, 3:5, 3:5] = g[:, None, None] * (d[:, :, None] * d[:, None, :] / h ** 4 - eye[None, :, :] / h ** 2)

    # 8. ant-rock Gaussian
    d = x - ROCK_POS
    h = ROCK_FEATURE_H_ANT
    g = np.exp(-np.sum(d * d, axis=1) / (2 * h * h))
    val[:, 8] = g
    grad[:, 8, :2] = -g[:, None] * d / h ** 2
    H[:, 8, :2, :2] = g[:, None, None] * (d[:, :, None] * d[:, None, :] / h ** 4 - eye[None, :, :] / h ** 2)

    return val, grad, H

def law_moments(states):
    return zeta_values_grad_hess(states)[0].mean(axis=0)

def critic_basis_and_dm(m):
    m = np.atleast_2d(m)
    G, K = m.shape
    phi = np.empty((G, N_CRITIC_BASIS))
    dp = np.zeros((G, N_CRITIC_BASIS, K))
    idx = 0
    phi[:, idx] = 1.0
    idx += 1
    for k in range(K):
        phi[:, idx] = m[:, k]
        dp[:, idx, k] = 1.0
        idx += 1
    for i in range(K):
        for j in range(i, K):
            phi[:, idx] = m[:, i] * m[:, j]
            dp[:, idx, i] += m[:, j]
            dp[:, idx, j] += m[:, i]
            idx += 1
    return phi, dp

def critic_value_lions(states, m, theta):
    st = np.atleast_2d(states)
    mm = np.atleast_2d(m)
    if mm.shape[0] == 1 and st.shape[0] > 1:
        mm = np.repeat(mm, st.shape[0], axis=0)
    phi, dp = critic_basis_and_dm(mm)
    dV = np.einsum('gnk,n->gk', dp, theta)
    _, gz, Hz = zeta_values_grad_hess(st)
    return phi @ theta, np.einsum('gk,gka->ga', dV, gz), np.einsum('gk,gkab->gab', dV, Hz)

def derivative_self_check():
    s = np.array([[0.3, -0.4, 0.7, 1.2, 0.5]])
    _, g, H = zeta_values_grad_hess(s)
    h = 2e-5
    max_grad_err = 0.0
    max_hess_err = 0.0
    for k in range(N_MOMENTS):
        gf = np.zeros(5)
        Hf = np.zeros((5, 5))
        for a in range(5):
            sp = s[0].copy()
            sm = s[0].copy()
            sp[a] += h
            sm[a] -= h
            gf[a] = (zeta_values_grad_hess(sp[None, :])[0][0, k] - zeta_values_grad_hess(sm[None, :])[0][0, k]) / (2 * h)
            gp = zeta_values_grad_hess(sp[None, :])[1][0, k]
            gm = zeta_values_grad_hess(sm[None, :])[1][0, k]
            Hf[:, a] = (gp - gm) / (2 * h)
        max_grad_err = max(max_grad_err, np.max(np.abs(gf - g[0, k])))
        max_hess_err = max(max_hess_err, np.max(np.abs(Hf - H[0, k])))
    print(f'  derivative check: max grad err={max_grad_err:.2e}, max Hess err={max_hess_err:.2e}')
    if max_grad_err > 2e-5 or max_hess_err > 2e-4:
        raise RuntimeError('Derivative check failed.')

# =============================================================================
# DATA COLLECTION
# =============================================================================
@dataclass
class MFData:
    s: np.ndarray
    sp: np.ndarray
    raw_u: np.ndarray
    raw_eta: np.ndarray
    u: np.ndarray
    eta: np.ndarray
    reward: np.ndarray
    r_lambda: np.ndarray
    old_logp: np.ndarray
    group_id: np.ndarray
    moments: np.ndarray
    group_times: np.ndarray
    group_weights: np.ndarray
    n_groups: int
    n_particles: int
    dt: float

def collect_data(omega, rng):
    M = N_PART_PHIBE
    G = L_EPS_DATA * N_STEPS_DATA
    N = G * M
    s = np.empty((N, 5))
    sp = np.empty((N, 5))
    ru = np.empty((N, 2))
    re = np.empty(N)
    ua = np.empty((N, 2))
    ea = np.empty(N)
    rr = np.empty(N)
    rl = np.empty(N)
    lp = np.empty(N)
    gid = np.empty(N, dtype=np.int64)
    mom = np.empty((G, N_MOMENTS))
    times = np.empty(G)

    row = 0
    g = 0
    for ep in range(L_EPS_DATA):
        y = COOKIE_START.copy()
        ang = rng.uniform(0, 2 * np.pi, M)
        rad = rng.uniform(0.4, ANT_INIT_R, M)
        x = y + np.stack([rad * np.cos(ang), rad * np.sin(ang)], axis=1)
        z = rng.normal(0, 1, M)

        for j in range(N_STEPS_DATA):
            states = np.column_stack([x, z, np.tile(y, (M, 1))])
            mom[g] = law_moments(states)
            times[g] = j * DT_DATA

            u, eta, rui, rei, logp = sample_policy_np(x, z, y, omega, rng)
            reward = running_reward_all(x, z, y, u, eta)
            rlam = reward - LAMBDA_ENT * logp

            branch_sp = local_frozen_action_branches(x, z, y, u, eta, rng, DT_DATA)
            xn, zn, yn = advance_occupancy(x, z, y, u, eta, rng, DT_DATA)

            sl = slice(row, row + M)
            s[sl] = states
            sp[sl] = branch_sp
            ru[sl] = rui
            re[sl] = rei
            ua[sl] = u
            ea[sl] = eta
            rr[sl] = reward
            rl[sl] = rlam
            lp[sl] = logp
            gid[sl] = g

            x, z, y = xn, zn, yn
            row += M
            g += 1

    w = np.exp(-BETA_DISC * times) if USE_DISCOUNTED_WEIGHTS else np.ones(G)
    w /= w.sum()
    return MFData(s, sp, ru, re, ua, ea, rr, rl, lp, gid, mom, times, w, G, M, DT_DATA)

# =============================================================================
# CRITIC + ADVANTAGE
# =============================================================================
def one_step_base_generator(data):
    d = data.sp - data.s
    bh = d / data.dt
    _, g, H = zeta_values_grad_hess(data.s)
    return np.einsum('na,nka->nk', bh, g) + 0.5 * np.einsum('na,nkab,nb->nk', d, H, d) / data.dt

def solve_critic(data):
    G, M = data.n_groups, data.n_particles
    phi, dp = critic_basis_and_dm(data.moments)
    gb = one_step_base_generator(data)
    gbar = gb.reshape(G, M, N_MOMENTS).mean(axis=1)
    rbar = data.r_lambda.reshape(G, M).mean(axis=1)
    Lphi = np.einsum('gnk,gk->gn', dp, gbar)
    R = BETA_DISC * phi - Lphi
    w = data.group_weights
    A = phi.T @ (w[:, None] * R)
    b = phi.T @ (w * rbar)
    Ar = A + CRITIC_RIDGE * np.eye(N_CRITIC_BASIS)
    cond = np.linalg.cond(Ar)
    if np.isfinite(cond) and cond < 1e10:
        theta = np.linalg.solve(Ar, b)
    else:
        theta = np.linalg.lstsq(Ar, b, rcond=1e-10)[0]
    return theta, {'phi': phi, 'dp': dp, 'gb': gb, 'cond': float(cond)}

def advantages(data, theta, diag):
    G, M = data.n_groups, data.n_particles
    phi, dp = diag['phi'], diag['dp']
    V = phi @ theta
    dV = np.einsum('gnk,n->gk', dp, theta)
    dVs = dV[data.group_id]
    _, gz, Hz = zeta_values_grad_hess(data.s)
    lg = np.einsum('nk,nka->na', dVs, gz)
    lH = np.einsum('nk,nkab->nab', dVs, Hz)
    d = data.sp - data.s
    bh = d / data.dt
    gen = np.einsum('na,na->n', bh, lg) + 0.5 * np.einsum('na,nab,nb->n', d, lH, d) / data.dt
    q = data.r_lambda + gen - BETA_DISC * V[data.group_id]

    # identity check against base generator on zetas
    fast = np.sum(diag['gb'] * dVs, axis=1)
    ident = float(np.max(np.abs(gen - fast)))

    a = q.copy()
    if CENTER_ADV_BY_LAW:
        a -= np.repeat(a.reshape(G, M).mean(axis=1), M)
    if NORMALIZE_ADV_GLOBALLY and np.std(a) > 1e-12:
        a /= np.std(a)
    return q, a, ident

# =============================================================================
# PPO
# =============================================================================
def ppo_update(data, adv_np, omega_old_np):
    st = torch.as_tensor(data.s)
    ru = torch.as_tensor(data.raw_u)
    re = torch.as_tensor(data.raw_eta)
    oldlp = torch.as_tensor(data.old_logp)
    adv = torch.as_tensor(adv_np)
    wf = torch.as_tensor(data.group_weights[data.group_id] / data.n_particles)
    old = torch.as_tensor(omega_old_np)

    omega = torch.nn.Parameter(old.clone())
    opt = torch.optim.Adam([omega], lr=PPO_LR)
    N = st.shape[0]
    rng = np.random.default_rng(12345)
    klv = 0.0
    clipf = 0.0
    epochs = 0

    for e in range(PPO_EPOCHS):
        perm = rng.permutation(N)
        for start in range(0, N, PPO_MINIBATCH):
            ix = torch.as_tensor(perm[start:start + PPO_MINIBATCH], dtype=torch.long)
            nlp = torch_logp(st[ix], ru[ix], re[ix], omega)
            ratio = torch.exp(torch.clamp(nlp - oldlp[ix], -20, 20))
            s1 = ratio * adv[ix]
            s2 = torch.clamp(ratio, 1 - PPO_CLIP_EPS, 1 + PPO_CLIP_EPS) * adv[ix]
            ww = wf[ix]
            obj = torch.sum(ww * torch.minimum(s1, s2)) / torch.sum(ww)
            loss = -obj
            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_([omega], PPO_MAX_GRAD_NORM)
            opt.step()

        with torch.no_grad():
            klv = float(torch.sum(wf * torch_kl(st, old, omega)) / torch.sum(wf))
            rat = torch.exp(torch.clamp(torch_logp(st, ru, re, omega) - oldlp, -20, 20))
            clipf = float(torch.sum(wf * ((rat < 1 - PPO_CLIP_EPS) | (rat > 1 + PPO_CLIP_EPS)).double()) / torch.sum(wf))
        epochs = e + 1
        if klv > PPO_TARGET_KL:
            break

    return omega.detach().numpy(), {'kl': klv, 'clip': clipf, 'epochs': epochs}

# =============================================================================
# TRAINING LOOP
# =============================================================================
def run_mf_phibe_ppo():
    if RUN_DERIVATIVE_CHECKS:
        derivative_self_check()

    rng = np.random.default_rng(SEED_DATA)
    omega = OMEGA_INIT.copy()
    omega_hist = [omega.copy()]
    theta_hist = []
    norm_hist = []
    value0_hist = []
    kl_hist = []

    print('\n' + '=' * 74)
    print(' law-based MF-PhiBE + PPO — Ant Colony Cookie Transport (T = 10)')
    print('=' * 74)
    print(f' critic V(mu): K={N_MOMENTS}, basis size={N_CRITIC_BASIS}')
    print(f' Delta t={DT_DATA}, T_data={T_DATA}, steps/episode={N_STEPS_DATA}, samples/update={L_EPS_DATA * N_STEPS_DATA * N_PART_PHIBE}')
    print(f' TIME_SCALE={TIME_SCALE}, beta={BETA_DISC}, C_Y={C_Y}, PG iterations={N_POLICY_ITER}')

    for k in range(N_POLICY_ITER):
        data = collect_data(omega, rng)
        theta, diag = solve_critic(data)
        q, adv, ident = advantages(data, theta, diag)
        omega_new, pd = ppo_update(data, adv, omega)
        first = np.arange(0, data.n_groups, N_STEPS_DATA)
        V0 = float(np.mean(critic_basis_and_dm(data.moments[first])[0] @ theta))
        theta_hist.append(theta.copy())
        norm_hist.append(np.linalg.norm(theta))
        value0_hist.append(V0)
        kl_hist.append(pd['kl'])
        print(
            f' [{k + 1:02d}/{N_POLICY_ITER}] '
            f'omega {np.round(omega, 3)} -> {np.round(omega_new, 3)} '
            f'| cond={diag["cond"]:.2e} | V(mu0)={V0:.2f} '
            f'| qstd={np.std(q):.2f} | KL={pd["kl"]:.3e} '
            f'| clip={pd["clip"]:.2f} | id={ident:.1e}'
        )
        omega = omega_new
        omega_hist.append(omega.copy())

    return omega, np.array(omega_hist), np.array(theta_hist), np.array(norm_hist), np.array(value0_hist), np.array(kl_hist)

# =============================================================================
# ANIMATION
# =============================================================================
def simulate_animation(omega, seed=SEED_ANIM):
    rng = np.random.default_rng(seed)
    n_sim = int(round(T_ANIM / DT_ANIM))
    n_fr = n_sim // SIM_PER_FRAME
    ctrl = max(1, int(round(DT_DATA / DT_ANIM)))

    y = COOKIE_START.copy()
    ang = rng.uniform(0, 2 * np.pi, N_ANTS)
    rad = rng.uniform(0.3, ANT_INIT_R, N_ANTS)
    x = y + np.stack([rad * np.cos(ang), rad * np.sin(ang)], axis=1)
    z = rng.normal(-1.0, 1.0, N_ANTS)
    u = np.zeros((N_ANTS, 2))
    eta = np.zeros(N_ANTS)

    hx = np.empty((n_fr, N_ANTS, 2))
    hz = np.empty((n_fr, N_ANTS))
    hy = np.empty((n_fr, 2))
    ht = np.empty(n_fr)
    frame = 0

    for step in range(n_sim):
        if step % ctrl == 0:
            u, eta, _, _, _ = sample_policy_np(x, z, y, omega, rng)
        if step % SIM_PER_FRAME == 0 and frame < n_fr:
            hx[frame] = x
            hz[frame] = z
            hy[frame] = y
            ht[frame] = step * DT_ANIM
            frame += 1
        x, z, y = environment_substep(x, z, y, u, eta, rng, DT_ANIM)

    print(f' animation: final cookie={np.round(hy[-1], 3)}, dist(target)={np.linalg.norm(hy[-1] - TARGET):.3f}')
    return hx, hz, hy, ht

# =============================================================================
# DRAWING UTILITIES
# =============================================================================
def draw_cookie(ax, y, alpha=1.0):
    r = 0.42
    ax.add_patch(mpatches.Circle(y, r, color=COL_COOKIE, zorder=5, alpha=alpha))
    ax.add_patch(mpatches.Circle(y, r, fill=False, edgecolor=COL_CRUMB, linewidth=1.8, zorder=6, alpha=alpha))
    for ox, oy in [(0.0, 0.15), (0.13, -0.08), (-0.13, -0.08), (0.0, -0.22), (0.18, 0.12), (-0.18, 0.12)]:
        ax.add_patch(mpatches.Circle((y[0] + ox, y[1] + oy), 0.055, color=COL_CRUMB, zorder=7, alpha=alpha))

def _rock_vertices():
    rg = np.random.default_rng(99)
    nv = 10
    an = np.linspace(0, 2 * np.pi, nv, endpoint=False) + rg.uniform(-0.18, 0.18, nv)
    rr = ROCK_RADIUS * rg.uniform(0.75, 1.25, nv)
    return ROCK_POS + np.stack([rr * np.cos(an), rr * np.sin(an)], axis=1)

def draw_rock(ax, label=True, alpha=1.0):
    v = _rock_vertices()
    ax.add_patch(plt.Polygon(v, facecolor='#888888', edgecolor='#555555', linewidth=1.8, zorder=4, alpha=0.92 * alpha))
    ax.add_patch(plt.Polygon(v[:4], facecolor='#AAAAAA', edgecolor='none', zorder=5, alpha=0.45 * alpha))
    ax.add_patch(plt.Polygon(v[5:], facecolor='#555555', edgecolor='none', zorder=5, alpha=0.30 * alpha))
    if label:
        ax.text(ROCK_POS[0], ROCK_POS[1] - ROCK_RADIUS - 0.22, 'rock', ha='center', fontsize=7.5, color='#555555', style='italic', zorder=6)

def draw_target(ax, label=True):
    ax.plot(*TARGET, marker='*', ms=22, color=COL_TARGET, markeredgecolor='white', markeredgewidth=1.2, zorder=10)
    if label:
        ax.text(TARGET[0] + 0.18, TARGET[1] + 0.3, 'TARGET', color=COL_TARGET, fontsize=9, fontweight='bold', zorder=10)

def draw_ant_professional(ax, center, direction, scale=1.0, color_body='#2B231D', color_outline='#120F0D', zorder=8):
    d = direction / (np.linalg.norm(direction) + 1e-12)
    p = np.array([-d[1], d[0]])

    abdomen_c = center - 0.12 * scale * d
    thorax_c = center + 0.00 * scale * d
    head_c = center + 0.12 * scale * d

    abdomen = mpatches.Ellipse(abdomen_c, width=0.18 * scale, height=0.28 * scale,
                               angle=np.degrees(np.arctan2(d[1], d[0])) - 90,
                               facecolor=color_body, edgecolor=color_outline, linewidth=0.6, zorder=zorder)
    thorax = mpatches.Ellipse(thorax_c, width=0.15 * scale, height=0.22 * scale,
                              angle=np.degrees(np.arctan2(d[1], d[0])) - 90,
                              facecolor=color_body, edgecolor=color_outline, linewidth=0.6, zorder=zorder + 0.1)
    head = mpatches.Ellipse(head_c, width=0.11 * scale, height=0.15 * scale,
                            angle=np.degrees(np.arctan2(d[1], d[0])) - 90,
                            facecolor=color_body, edgecolor=color_outline, linewidth=0.6, zorder=zorder + 0.2)
    ax.add_patch(abdomen)
    ax.add_patch(thorax)
    ax.add_patch(head)

    # antennae
    a0 = head_c + 0.08 * scale * d
    for sgn in (-1, 1):
        a1 = a0 + 0.05 * scale * d + sgn * 0.05 * scale * p
        ax.plot([a0[0], a1[0]], [a0[1], a1[1]], color=color_outline, lw=0.55, zorder=zorder)

    # 3 pairs of legs from thorax
    bases = [thorax_c - 0.04 * scale * d, thorax_c, thorax_c + 0.04 * scale * d]
    for base in bases:
        for sgn in (-1, 1):
            knee = base + sgn * 0.08 * scale * p - 0.01 * scale * d
            foot = knee + sgn * 0.05 * scale * p - 0.03 * scale * d
            ax.plot([base[0], knee[0], foot[0]], [base[1], knee[1], foot[1]], color=color_outline, lw=0.55, zorder=zorder - 0.1)

def scene_directions(x, z, y):
    lam = Lambda(z)
    dc = y[None, :] - x
    rr = np.linalg.norm(dc, axis=1, keepdims=True)
    ds = dc / (rr + EPS)
    tv = TARGET - y
    dp = tv / (np.linalg.norm(tv) + EPS)
    dirs = (1 - lam[:, None]) * ds + lam[:, None] * dp[None, :]
    norms = np.linalg.norm(dirs, axis=1, keepdims=True)
    dirs = dirs / (norms + EPS)
    attached = lam > 0.5
    return lam, dirs, attached

# =============================================================================
# RENDERERS
# =============================================================================
def render_full_frame(i, x, z, y, t, trail, omega):
    fig = plt.figure(figsize=(11, 7), facecolor=COL_BG)
    ax = fig.add_axes([0.01, 0.04, 0.76, 0.92])
    ap = fig.add_axes([0.80, 0.04, 0.19, 0.92])
    ax.set_facecolor(COL_BG)
    ax.set_xlim(X_MIN, X_MAX)
    ax.set_ylim(Y_MIN, Y_MAX)
    ax.set_aspect('equal')
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
    for spine in ax.spines.values():
        spine.set_edgecolor(COL_GRID)
    for gx in np.arange(X_MIN, X_MAX + 0.5, 1.0):
        ax.axvline(gx, color=COL_GRID, lw=0.7, alpha=0.8)
    for gy in np.arange(Y_MIN, Y_MAX + 0.5, 1.0):
        ax.axhline(gy, color=COL_GRID, lw=0.7, alpha=0.8)

    if len(trail) > 1:
        seg = [trail[j:j + 2] for j in range(len(trail) - 1)]
        ax.add_collection(LineCollection(seg, colors=[COL_TRAIL] * len(seg), linewidths=2.5, alpha=0.5, zorder=2))

    draw_cookie(ax, y)
    dist = np.linalg.norm(y - TARGET)
    ax.text(y[0], y[1] - 0.68, f'd={dist:.2f}', ha='center', fontsize=7.5, color=COL_COOKIE, fontweight='bold', zorder=8)
    draw_target(ax, label=True)
    draw_rock(ax, label=True)

    lam, dirs, attached = scene_directions(x, z, y)
    if attached.any():
        ax.add_collection(LineCollection([np.array([x[j], y]) for j in np.where(attached)[0]], colors='#000000', linewidths=0.6, alpha=0.30, zorder=3))

    # ant bodies + color-coded attachment state
    for j in range(len(x)):
        ang = np.degrees(np.arctan2(dirs[j, 1], dirs[j, 0]))
        ax.add_patch(mpatches.Ellipse(x[j], width=0.08, height=0.27, angle=ang - 90,
                                      facecolor='#AAAAAA', edgecolor='#777777', linewidth=0.55, alpha=0.38, zorder=7))
    ax.scatter(x[:, 0], x[:, 1], c=lam, cmap=CMAP_ANT, vmin=0, vmax=1, s=34, zorder=8, edgecolors='#444444', linewidths=0.35)
    hp = x + 0.15 * dirs
    ax.scatter(hp[:, 0], hp[:, 1], c='#111111', s=7, zorder=9)
    ax.add_patch(mpatches.Circle(y, ELL, fill=False, ls='--', edgecolor='#AAAAAA', lw=0.8, alpha=0.4, zorder=2))
    ax.set_title(f'Ant Colony Cookie Transport — MF-PhiBE + PPO  (t={t:.1f}s)   Attached: {int(attached.sum())}/{N_ANTS}',
                 fontsize=12, fontweight='bold', color='#333333')

    ap.set_facecolor('#EDD9B0')
    ap.axis('off')
    ap.set_xlim(0, 1)
    ap.set_ylim(0, 1)

    def info(yy, label, val, color='#333333'):
        ap.text(0.08, yy, label, fontsize=9, color='#555555')
        ap.text(0.92, yy, val, fontsize=9, color=color, fontweight='bold', ha='right')

    ap.text(0.5, 0.95, 'MF-PhiBE Stats', ha='center', fontsize=11, fontweight='bold')
    info(0.84, 'Time (s)', f'{t:.1f}')
    info(0.75, 'Cookie→target', f'{dist:.3f}', '#882222')
    info(0.66, 'Attached', f'{int(attached.sum())}/{N_ANTS}')
    info(0.57, 'Mean Λ(z)', f'{lam.mean():.3f}')
    info(0.43, 'ω₁ seek', f'{omega[0]:.3f}', '#225588')
    info(0.34, 'ω₂ pull', f'{omega[1]:.3f}', '#225588')
    info(0.25, 'ω₃ eta', f'{omega[2]:.3f}', '#225588')
    info(0.11, 'Δt (PhiBE)', f'{DT_DATA:.3f}', '#225588')

    p = FULL_FRAME_DIR / f'frame_{i:04d}.png'
    fig.savefig(p, dpi=90, bbox_inches='tight', facecolor=COL_BG)
    plt.close(fig)
    return str(p)

def render_simple_frame(i, x, z, y, t, trail):
    fig = plt.figure(figsize=(7.4, 5.2), facecolor='#FFFFFF')
    ax = fig.add_axes([0.035, 0.07, 0.93, 0.87])
    ax.set_facecolor('#FFFFFF')
    ax.set_xlim(X_MIN, X_MAX)
    ax.set_ylim(Y_MIN, Y_MAX)
    ax.set_aspect('equal')
    ax.axis('off')

    # subtle background
    for gx in np.arange(X_MIN, X_MAX + 1.0, 1.5):
        ax.axvline(gx, color='#FFFFFF', lw=0.6, alpha=0.0, zorder=0)
    for gy in np.arange(Y_MIN, Y_MAX + 1.0, 1.5):
        ax.axhline(gy, color='#FFFFFF', lw=0.6, alpha=0.0, zorder=0)

    if len(trail) > 1:
        seg = [trail[j:j + 2] for j in range(len(trail) - 1)]
        ax.add_collection(LineCollection(seg, colors=['#C4A063'] * len(seg), linewidths=2.0, alpha=0.35, zorder=1))

    # target + obstacle without labels
    draw_target(ax, label=False)
    draw_rock(ax, label=False, alpha=0.95)
    draw_cookie(ax, y, alpha=1.0)

    lam, dirs, attached = scene_directions(x, z, y)
    if attached.any():
        ax.add_collection(LineCollection([np.array([x[j], y]) for j in np.where(attached)[0]], colors='#201A16', linewidths=0.45, alpha=0.18, zorder=2))

    # more ant-like, uniform professional style
    for j in range(len(x)):
        draw_ant_professional(ax, x[j], dirs[j], scale=0.6)

    # elegant time tag only
    tag = mpatches.FancyBboxPatch((0.40, 0.945), 0.20, 0.045,
                                  boxstyle='round,pad=0.01,rounding_size=0.015',
                                  transform=ax.transAxes, facecolor='white', edgecolor='#DDD6C7',
                                  linewidth=0.9, alpha=0.96, zorder=20)
    ax.add_patch(tag)
    ax.text(0.50, 0.968, f't = {t:4.1f} s', transform=ax.transAxes,
            ha='center', va='center', fontsize=10.2, color='#3B342E', fontweight='semibold', zorder=21)

    p = SIMPLE_FRAME_DIR / f'frame_{i:04d}.png'
    fig.savefig(p, dpi=100, bbox_inches='tight', facecolor='#FFFFFF')
    plt.close(fig)
    return str(p)

# =============================================================================
# OUTPUT / COMPILATION
# =============================================================================
def save_convergence(norms, omega_hist, value0_hist, kl_hist):
    fig, ax = plt.subplots(1, 2, figsize=(10, 4))
    fig.suptitle('MF-PhiBE + PPO Convergence — Ant Colony Cookie Transport', fontweight='bold')
    it = np.arange(1, len(norms) + 1)
    ax[0].plot(it, norms, 'o-', lw=2)
    ax[0].set_xlabel('Policy iteration')
    ax[0].set_ylabel(r'$\|\theta^*\|_2$')
    ax[0].set_title('Law-space Critic Norm')
    ax[0].grid(alpha=0.35, ls='--')

    io = np.arange(len(omega_hist))
    ax[1].plot(io, omega_hist[:, 0], 'o-', label=r'$\omega_1$ seek')
    ax[1].plot(io, omega_hist[:, 1], 's-', label=r'$\omega_2$ pull')
    ax[1].plot(io, omega_hist[:, 2], '^-', label=r'$\omega_3$ eta')
    ax[1].set_xlabel('Policy iteration')
    ax[1].set_title('PPO Policy Parameters')
    ax[1].legend()
    ax[1].grid(alpha=0.35, ls='--')
    fig.tight_layout()
    p = OUT_DIR / 'mf_phibe_ppo_convergence.png'
    fig.savefig(p, dpi=140)
    plt.close(fig)

    np.savetxt(
        OUT_DIR / 'training_diagnostics.csv',
        np.column_stack([it, norms, value0_hist, kl_hist]),
        delimiter=',',
        header='iteration,theta_norm,V_mu0,ppo_KL',
        comments=''
    )

def compile_gif(frame_paths, out_path, fps):
    frames = [Image.open(p).convert('P', palette=Image.ADAPTIVE, colors=128) for p in frame_paths]
    frames[0].save(out_path, save_all=True, append_images=frames[1:], optimize=True,
                   duration=int(1000 / fps), loop=0)

def compile_mp4(frame_paths, out_path, fps, frame_dir):
    if not shutil.which('ffmpeg'):
        return False
    lst = frame_dir / 'list.txt'
    lst.write_text(''.join(f"file '{Path(p).resolve()}'\nduration {1 / fps:.4f}\n" for p in frame_paths))
    r = subprocess.run([
        'ffmpeg', '-y', '-f', 'concat', '-safe', '0', '-i', str(lst),
        '-vf', 'scale=trunc(iw/2)*2:trunc(ih/2)*2',
        '-c:v', 'libx264', '-pix_fmt', 'yuv420p', str(out_path)
    ], capture_output=True)
    return r.returncode == 0

# =============================================================================
# MAIN
# =============================================================================
def main():
    clean_frame_dir(FULL_FRAME_DIR)
    clean_frame_dir(SIMPLE_FRAME_DIR)

    omega, omega_hist, theta_hist, norm_hist, value0_hist, kl_hist = run_mf_phibe_ppo()
    save_convergence(norm_hist, omega_hist, value0_hist, kl_hist)

    hx, hz, hy, ht = simulate_animation(omega)
    trail = [hy[i] for i in range(len(ht))]

    full_paths = []
    simple_paths = []
    print(f' rendering {len(ht)} frames for both GIFs ...')
    for i in range(len(ht)):
        if i % 20 == 0:
            print(f'   frame {i + 1}/{len(ht)} | t={ht[i]:.1f}s | dist={np.linalg.norm(hy[i] - TARGET):.2f}')
        full_paths.append(render_full_frame(i, hx[i], hz[i], hy[i], ht[i], trail[:i + 1], omega))
        simple_paths.append(render_simple_frame(i, hx[i], hz[i], hy[i], ht[i], trail[:i + 1]))

    full_gif = OUT_DIR / f'ant_colony_mf_phibe_ppo_full_iter{N_POLICY_ITER}_T10.gif'
    simple_gif = OUT_DIR / f'ant_colony_mf_phibe_ppo_simple_iter{N_POLICY_ITER}_T10.gif'
    compile_gif(full_paths, full_gif, FPS_FULL)
    compile_gif(simple_paths, simple_gif, FPS_SIMPLE)

    full_mp4 = OUT_DIR / f'ant_colony_mf_phibe_ppo_full_iter{N_POLICY_ITER}_T10.mp4'
    simple_mp4 = OUT_DIR / f'ant_colony_mf_phibe_ppo_simple_iter{N_POLICY_ITER}_T10.mp4'
    ok_full = compile_mp4(full_paths, full_mp4, FPS_FULL, FULL_FRAME_DIR)
    ok_simple = compile_mp4(simple_paths, simple_mp4, FPS_SIMPLE, SIMPLE_FRAME_DIR)

    np.savez(
        OUT_DIR / 'learned_mf_phibe_ppo_T10.npz',
        omega_final=omega,
        omega_hist=omega_hist,
        theta_hist=theta_hist,
        norm_theta_hist=norm_hist,
        value0_hist=value0_hist,
        kl_hist=kl_hist,
        hist_y=hy,
        hist_t=ht,
    )

    print('\n' + '=' * 74)
    print('DONE')
    print('=' * 74)
    print('final omega =', np.round(omega, 5))
    print('full GIF   =', full_gif)
    print('simple GIF =', simple_gif)
    print('full MP4   =', full_mp4 if ok_full else '(ffmpeg unavailable/failed)')
    print('simple MP4 =', simple_mp4 if ok_simple else '(ffmpeg unavailable/failed)')

if __name__ == '__main__':
    main()

  derivative check: max grad err=3.64e-11, max Hess err=2.39e-11

 law-based MF-PhiBE + PPO — Ant Colony Cookie Transport (T = 10)
 critic V(mu): K=9, basis size=55
 Delta t=0.05, T_data=10.0, steps/episode=200, samples/update=9600
 TIME_SCALE=2.2, beta=0.3, C_Y=5.0, PG iterations=18
 [01/18] omega [0.2 0.2 0.2] -> [0.215 0.244 0.142] | cond=1.60e+06 | V(mu0)=-148.65 | qstd=140.03 | KL=9.099e-03 | clip=0.13 | id=3.4e-13
 [02/18] omega [0.215 0.244 0.142] -> [0.107 0.202 0.152] | cond=2.09e+06 | V(mu0)=-155.61 | qstd=359.73 | KL=7.217e-03 | clip=0.08 | id=9.1e-13
 [03/18] omega [0.107 0.202 0.152] -> [0.056 0.121 0.144] | cond=1.69e+06 | V(mu0)=-166.03 | qstd=230.59 | KL=6.452e-03 | clip=0.07 | id=4.5e-13
 [04/18] omega [0.056 0.121 0.144] -> [-0.017  0.076  0.211] | cond=1.74e+06 | V(mu0)=-180.09 | qstd=251.95 | KL=1.134e-02 | clip=0.18 | id=9.1e-13
 [05/18] omega [-0.017  0.076  0.211] -> [0.067 0.188 0.221] | cond=1.54e+06 | V(mu0)=-150.42 | qstd=86.54 | KL=1.117e-02 | clip=0.15 | id